### RecursiveCharacterTextSplitter

- Document를 설정한 chunk를 기준으로 분리


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
        """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

0 : 환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


### Vector DB

- TextSplitter로 분리한 데이터를 Embedding 처리한 후 저장


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

API_KEY = os.getenv("nvidiaapi_key")
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

embedding = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("documents : ", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

print("chunk : ", len(chunks))

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

print("FIASS 생성완료")

query = "펀드가 무엇인가요?"

result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print("결과 : ", i)
    print("페이지 : ", doc.metadata.get("page"))
    print()
    print(doc.page_content)
    print()

### RecursiveCharacterTextSplitter 및 Vector DB로 LLM까지 연결


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(api_key=API_KEY, model=MODEL)

embeddings = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]{question}",
        ),
    ]
)

parser = StrOutputParser()

# question = "펀드란 무엇인가요?"
question = "대한민국의 수도는 어디인가요?"

retriever_docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in retriever_docs])

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

### Vector DB에서 similarity_search_with_score로 통한 유사도 점수 확인


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

questions = [
    "펀드란 무엇인가요?",
    "대한민국의 수도는 어디인가요?",
]

for question in questions:
    result = vectorstore.similarity_search_with_score(question, k=3)

    print("질문 : ", question)

    for i, (doc, score) in enumerate(result):
        print("검색 결과")
        print("distance : ", score)
        print("page : ", doc.metadata.get("page"))
        print("내용 : ", doc.page_content[:500])

### MMR(Maximum Marginal Relevance)

- 질문과 관련 있으면서 서로 다른 정보를 가진 문서를 검색


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드의 종류와 특징은 무엇인가요?"

similarity_retriever = vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)

similarity_docs = similarity_retriever.invoke(question)

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5,
    },
)

mmr_docs = mmr_retriever.invoke(question)

print("\n")
print("=" * 80)
print("Similarity Search 결과")
print("=" * 80)

for i, doc in enumerate(similarity_docs):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print(doc.page_content[:500])


# ============================================================
# 11. MMR 결과
# ============================================================

print("\n")
print("=" * 80)
print("MMR Search 결과")
print("=" * 80)

for i, doc in enumerate(mmr_docs):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print(doc.page_content[:500])

### Metadata Filtering

- Vector DB에서 검색하기 전에 특정 조건에 맞는 문서만 검색 대상으로 제한


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)


documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드란 무엇인가?"

all_results = vectorstore.similarity_search(question, k=3)

for i, doc in enumerate(all_results):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print("Content:")
    print(doc.page_content[:500])

filtered_results = vectorstore.similarity_search(question, k=3, filter={"page": 50})

print()

if not filtered_results:
    print("조건에 맞는 문서를 찾지 못했습니다.")

else:
    for i, doc in enumerate(filtered_results):
        print(f"\n----- 결과 {i + 1} -----")

        print("Page:", doc.metadata.get("page"))

        print("Content:")
        print(doc.page_content[:500])

### Hybrid Search

- Vector Search(의미가 비슷한 문서 반환)과 Keyword Search(정확한 단어로 검색)를 모두 활용


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"


llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)


vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 3

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

### Reranker

- 찾아온 문서들 중에서 질문과 관련 있는 순서대로 다시 정리
- 일반적으로 Retriever보다 계산량이 많기에 후보를 추려오면 그 뒤 사용


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA, NVIDIARerank
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
NVIDIA_RERANK_KEY = os.getenv("nvidiaapi_key")
NVIDIA_RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

reranker = NVIDIARerank(model=NVIDIA_RERANK_MODEL, api_key=NVIDIA_RERANK_KEY)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)

print("검색된 후보 문서 : ", len(retrieved_docs))

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

final_docs = reranked_docs[:3]

for i, doc in enumerate(final_docs, start=1):
    print(f"\n--- 최종 문서 {i} ---")

    print("page:", doc.metadata.get("page"))

    print(doc.page_content[:500])


context = "\n\n".join([doc.page_content for doc in final_docs])

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]\n{question}",
        ),
    ]
)

### Multi-Query

- 하나의 질문을 여러 개의 검색 질문으로 변환한 뒤 각각 검색하는 방법
- 벡터 검색은 의미가 비슷하면 어느 정도 찾아주지만, 한 번의 검색으로는 놓치는 문서가 생길 수 있습니다.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
NVIDIA_RERANK_KEY = os.getenv("nvidiaapi_key")
NVIDIA_RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwarg={"k": 3})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

prompt = ChatPromptTemplate.from_template("""
    다음 질문에 대해 문서 검색에 사용할 수 있는 서로 다른 검색 질문 3래를 만들어주세요.

    원래 질문
    {question}

    조건:
    - 원래 질문의 의미는 유지할 것
    - 서로 다른 표현을 사용할 것
    - 검색에 도움이 되는 핵심 단어를 포함할 것
    - 번호나 설명없이 검색 질문만 한 줄씩 출력할 것
""")

parser = StrOutputParser()

reranker = NVIDIARerank(model=NVIDIA_RERANK_MODEL, api_key=NVIDIA_RERANK_KEY)

chain = prompt | llm | parser

generated_queries_text = chain.invoke({"question": question})

generated_queries = [
    query.strip() for query in generated_queries_text.split("\n") if query.strip()
]

all_docs = []

for i, query in enumerate(generated_queries, start=1):
    print(query)
    docs = hybrid_retriever.invoke(query)

    print("검색된 문서 수:", len(docs))

    # 검색 결과를 하나의 리스트에 모읍니다.
    all_docs.extend(docs)


print("\n전체 검색 결과:", len(all_docs))

unique_docs = {}

for doc in all_docs:
    page = doc.metadata.get("page")

    if page not in unique_docs:
        unique_docs[page] = doc
unique_docs = list(unique_docs.values())

print("중복 제거 후 문서 수 : ", len(unique_docs))

reranked_docs = reranker.compress_documents(documents=unique_docs, query=question)

final_docs = reranked_docs[:3]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

### Query Transformation

- 사용자 질문을 그대로 검색하지 않고 검색하기 좋은 형태로 바꿔주는 것
- 1. Rewrite, Step-back, 3. HyDE 등


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"


llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwarg={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

question = "그거 모바일로 금융거래 할 때 조심해야 하는 게 뭐였지?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로 다시 작성하세요.

    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

rewrite_chain = rewrite_prompt | llm | parser

rewritten_question = rewrite_chain.invoke({"question": question})

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

final_docs = retrieved_docs[:3]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

### Context Compression

- 검색된 문서에서 질문에 필요한 부분만 뽑아내는 것


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
NVIDIA_RERANK_KEY = os.getenv("nvidiaapi_key")
NVIDIA_RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

reranker = NVIDIARerank(
    model=NVIDIA_RERANK_MODEL,
    api_key=NVIDIA_RERANK_KEY,
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로 다시 작성하세요.

    조건:
    - 원래 질문의 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

rewrite_chain = rewrite_prompt | llm | parser

rewritten_question = rewrite_chain.invoke({"question": question})

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

reranked_docs = reranked_docs[:5]

compressor = LLMChainExtractor.from_llm(llm)

compressed_docs = []

for doc in reranked_docs:
    compressed_doc = compressor.compress_documents(documents=[doc], query=question)

    compressed_docs.extend(compressed_doc)

final_docs = compressed_docs[:5]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

### Citation

- 답변의 근거가 어디인지 같이 보여주는 것


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
RERANK_API_KEY = os.getenv("nvidiaapi_key")
RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

reranker = NVIDIARerank(api_key=RERANK_API_KEY, model=RERANK_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로
    다시 작성하세요.

    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewrite_chain = rewrite_prompt | llm | parser

rewritten_question = rewrite_chain.invoke({"question": question})

print("Rewrite:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

reranked_docs = reranked_docs[:5]

context_parts = []

for doc in reranked_docs:
    page = doc.metadata.get("page", "unknown")
    source = doc.metadata.get("source", "unknown")
    context_parts.append(
        f"""
        [출처: p.{page}]
        [파일: {source}]
        {doc.page_content}
        """
    )


context = "\n\n".join(context_parts)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로
            질문에 답변하는 도우미입니다.

            반드시 제공된 Context만 근거로 답변하세요.

            답변에 사용한 문서의 출처를
            반드시 다음 형식으로 표시하세요.

            [출처: p.페이지번호]

            Context에 없는 내용은 추측하지 마세요.

            문서에서 답을 찾을 수 없다면:

            "제공된 문서에서 답을 찾을 수 없습니다."

            라고 답변하세요.

            [문서 Context]
            {context}
            """,
        ),
        (
            "user",
            """
            [질문]
            {question}
            """,
        ),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"question": question, "context": context})

print()
print("result : ", result)

### RAG Evaluation(평가)

- 필요한 문서를 제대로 가져왔는지 평가


In [ ]:
import os

from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import (
    ChatNVIDIA,
    NVIDIAEmbeddings,
    NVIDIARerank,
)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"

EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

NVIDIA_RERANK_KEY = os.getenv("nvidiaapi_key")
NVIDIA_RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"

llm = ChatNVIDIA(
    model=MODEL,
    api_key=API_KEY,
)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

reranker = NVIDIARerank(
    model=NVIDIA_RERANK_MODEL,
    api_key=NVIDIA_RERANK_KEY,
)


PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)


vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})


bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 10


hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
다음 사용자의 질문을 문서 검색에 적합한 형태로
다시 작성하세요.

조건:
- 질문의 원래 의미를 유지하세요.
- 불필요한 표현은 제거하세요.
- 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
- 질문 하나만 출력하세요.
- 설명이나 번호는 출력하지 마세요.

사용자 질문:
{question}
"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def evaluate_hit_rate(
    question,
    relevant_pages,
    k=5,
):
    rewritten_question = rewrite_chain.invoke({"question": question})

    retrieved_docs = hybrid_retriever.invoke(rewritten_question)

    reranked_docs = reranker.compress_documents(
        documents=retrieved_docs,
        query=question,
    )

    top_docs = reranked_docs[:k]

    retrieved_pages = [doc.metadata.get("page") for doc in top_docs]

    hit = any(page in relevant_pages for page in retrieved_pages)

    return hit, retrieved_pages, rewritten_question


evaluation_data = [
    {
        "question": "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?",
        "relevant_pages": [25, 16, 21],
    },
]
hits = []

for item in evaluation_data:
    hit, pages, rewritten_question = evaluate_hit_rate(
        question=item["question"],
        relevant_pages=item["relevant_pages"],
        k=5,
    )

    hits.append(int(hit))

    print("=" * 60)

    print("원래 질문:")
    print(item["question"])

    print("\nRewrite된 질문:")
    print(rewritten_question)

    print("\n정답 페이지:")
    print(item["relevant_pages"])

    print("\n검색된 Top 5 페이지:")
    print(pages)

    print("\nHit@5:")
    print(int(hit))

hit_rate = sum(hits) / len(hits)

print("\n" + "=" * 60)

print(f"최종 Hit Rate@5: {hit_rate:.4f}")

print("=" * 60)

### LLM-as-a-Judge

- LLM에게 다른 LLM의 답변을 채점 시키는 것

### Context Relevance

- 검색해서 가져온 Context가 질문에 관련되어 있는지 체크

### Faithfulness

- 환각(Hallucination)을 얼마나 하고 있는지 확인

### Answer Relevance

- 질문에 제대로 답변 했는지 확인


In [ ]:
import os

from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import (
    ChatNVIDIA,
    NVIDIAEmbeddings,
    NVIDIARerank,
)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"

EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

NVIDIA_RERANK_KEY = os.getenv("nvidiaapi_key")
NVIDIA_RERANK_MODEL = "nvidia/llama-3.2-nv-rerankqa-1b-v2"


llm = ChatNVIDIA(
    model=MODEL,
    api_key=API_KEY,
)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

reranker = NVIDIARerank(
    model=NVIDIA_RERANK_MODEL,
    api_key=NVIDIA_RERANK_KEY,
)

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
다음 사용자의 질문을 문서 검색에 적합한 형태로
다시 작성하세요.

조건:
- 질문의 원래 의미를 유지하세요.
- 불필요한 표현은 제거하세요.
- 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
- 질문 하나만 출력하세요.
- 설명이나 번호는 출력하지 마세요.

사용자 질문:
{question}
"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

반드시 아래 제공된 문서 내용만 근거로 답변하세요.

문서에 답이 없다면
"제공된 문서에서 답을 찾을 수 없습니다."
라고 답변하세요.

[문서 내용]
{context}
""",
        ),
        ("user", "[질문]\n{question}"),
    ]
)

answer_chain = answer_prompt | llm | StrOutputParser()

context_eval_prompt = ChatPromptTemplate.from_template(
    """
당신은 RAG 검색 품질 평가자입니다.

사용자 질문과 검색된 문서를 보고,
검색된 문서가 질문에 얼마나 관련 있는지 평가하세요.

점수:
0.0 = 전혀 관련 없음
0.5 = 일부 관련
1.0 = 매우 관련 있음

반드시 숫자 하나만 출력하세요.

[질문]
{question}

[Context]
{context}
"""
)

context_eval_chain = context_eval_prompt | llm | StrOutputParser()

faithfulness_prompt = ChatPromptTemplate.from_template(
    """
당신은 RAG 답변의 사실성을 평가하는 평가자입니다.

답변에 포함된 내용이 제공된 Context에 의해
뒷받침되는지를 평가하세요.

Context에 없는 내용을 답변이 추가했다면
점수를 낮게 주세요.

점수:
0.0 = 대부분 근거 없음
0.5 = 일부만 근거 있음
1.0 = 모든 내용이 Context에 근거함

반드시 숫자 하나만 출력하세요.

[Context]
{context}

[Answer]
{answer}
"""
)

faithfulness_chain = faithfulness_prompt | llm | StrOutputParser()

answer_relevance_prompt = ChatPromptTemplate.from_template(
    """
당신은 RAG 답변의 질문 관련성을 평가하는 평가자입니다.

사용자의 질문과 답변을 비교하여
답변이 질문에 얼마나 적절하게 답했는지 평가하세요.

점수:
0.0 = 질문에 답하지 못함
0.5 = 일부만 답함
1.0 = 질문에 정확하게 답함

반드시 숫자 하나만 출력하세요.

[질문]
{question}

[Answer]
{answer}
"""
)

answer_relevance_chain = answer_relevance_prompt | llm | StrOutputParser()

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewritten_question = rewrite_chain.invoke({"question": question})

print("=" * 60)

print("원래 질문:")
print(question)

print("\nRewrite 질문:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(
    documents=retrieved_docs,
    query=question,
)

reranked_docs = reranked_docs[:5]

context = "\n\n".join(doc.page_content for doc in reranked_docs)

answer = answer_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)


print("\n" + "=" * 60)

print("최종 Answer:")
print(answer)

context_score = context_eval_chain.invoke(
    {
        "question": question,
        "context": context,
    }
)

faithfulness_score = faithfulness_chain.invoke(
    {
        "context": context,
        "answer": answer,
    }
)

answer_relevance_score = answer_relevance_chain.invoke(
    {
        "question": question,
        "answer": answer,
    }
)

print("\n" + "=" * 60)

print("RAG Evaluation")

print("-" * 60)

print("Context Relevance:", context_score)

print("Faithfulness:", faithfulness_score)

print("Answer Relevance:", answer_relevance_score)

print("=" * 60)